# Chat-Schleife und Token-Budget

In diesem Notebook werden wir:
1. API-Anfragen mit und ohne Gesprächsverlauf vergleichen.
2. Dieselbe Eingabeaufforderung (Prompt) mehrfach ausführen und die Antworten vergleichen.
3. Tokens zählen und die Eingabekosten für beide Unternehmen schätzen.
4. Die API-Nutzung bei niedrigem und hohem Reasoning-Aufwand (Denkaufwand) vergleichen.

**Daten:** Wir verwenden `data/ikea_reviews.csv` und `data/mcdonalds_reviews.csv`, die bereits einmal mit [explore_yelp_data.ipynb](../01_Explore_Yelp_Data/explore_yelp_data.ipynb) entpackt wurden. Die Dateien werden wiederverwendet; eine erneute Datenaufbereitung ist nicht erforderlich.


## Bibliotheken und Einstellungen
`client.responses.create()`  aus der OpenAI-Bibliothek sendet eine Anfrage an das Modell und gibt dessen Antwort zurück.

Jeder Aufruf ist eine neue Anfrage. Wenn du eine Codezelle erneut ausführst, werden die darin enthaltenen Anfragen nochmals gesendet. Dabei können unterschiedliche Antworten entstehen.

In [1]:
# Libraries
import os
import platform
from datetime import datetime
from pathlib import Path
from platform import python_version

import pandas as pd
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

# Folder with the data (relative to the folder of this notebook)
DATA = Path("../data")

# Model and course price assumption in USD per 1 million input tokens
MODEL = "gpt-6-luna"
PRICE_INPUT_PER_M = 0.10

# The API key is a Codespaces secret; outside Codespaces it is read from .env
load_dotenv("../.env")

# The client sends our requests and retries when the rate limit is reached
client = OpenAI(max_retries=8)

# Show current working directory
print(os.getcwd())

/workspaces/llm_fundamentals/02_Chat_Loop_Tokens


## Schritt 1: Der API eine Anschlussfrage stellen

Wir senden zwei separate Anfragen.

Versteht das Modell, worauf sich das Wort «it» (es) in der zweiten Frage bezieht?

In [2]:
# First question
r1 = client.responses.create(
    model=MODEL,
    input=(
        "Name one typical dish in the IKEA restaurant. "
        "Answer in one sentence."
    ),
)
print("Answer 1:", r1.output_text)

# Follow-up question, sent as a separate request
r2 = client.responses.create(model=MODEL, input="Why is it so popular?")
print("Answer 2:", r2.output_text)

Answer 1: A typical dish at the IKEA restaurant is Swedish meatballs served with mashed potatoes and lingonberry sauce.
Answer 2: What are you referring to? I can explain why it’s popular once I know what “it” is.


Die zweite Anfrage enthält keine Informationen über die vorherige Frage.

Das Modell kann zwar eine plausible Bedeutung von «it» vermuten, hat die vorherige Anfrage aber nicht gesehen.

**Diese Anfragen sind unabhängig voneinander: Unser Code hat keinen Gesprächsverlauf übermittelt.**

## Schritt 2: Eine Nachrichtenliste speichern und erneut senden
Eine Chat-Anwendung wie ChatGPT speichert den gesamten Gesprächsverlauf in einer Liste und sendet diese bei jeder neuen Frage vollständig mit.

Jede Nachricht besitzt eine role (Rolle) und einen content (Inhalt).

Die möglichen Rollen sind: `role` (`system`, `user` oder `assistant`)  und ein `content`.
- system: Systemanweisungen
- user: Nachrichten des Benutzers
- assistant: Antworten des Modells

In [3]:
# The system message holds the rules for the whole conversation
RULES = "You answer questions about IKEA. Keep every answer to one sentence."
messages = [{"role": "system", "content": RULES}]

# The questions of our conversation
conversation = [
    "Name one typical dish in the IKEA restaurant.",
    "Why is it so popular?",
    "What was my first question?",
]

for question in conversation:
    # 1. Add the question to the list
    messages.append({"role": "user", "content": question})

    # 2. Send the whole list
    r = client.responses.create(model=MODEL, input=messages)

    # 3. Add the answer to the list
    messages.append({"role": "assistant", "content": r.output_text})

    print("Q:", question)
    print("A:", r.output_text)
    print("   input tokens of this request:", r.usage.input_tokens, "\n")

Q: Name one typical dish in the IKEA restaurant.
A: Swedish meatballs are a typical dish in the IKEA restaurant.
   input tokens of this request: 32 

Q: Why is it so popular?
A: IKEA’s Swedish meatballs are popular because they’re affordable, familiar comfort food and part of the store’s distinctive Swedish identity.
   input tokens of this request: 61 

Q: What was my first question?
A: Your first question was, “Name one typical dish in the IKEA restaurant.”
   input tokens of this request: 103 



Jetzt kann das Modell die Anschlussfragen beantworten, weil die vorherigen Fragen und Antworten Teil der Eingabe sind.

Der Nachteil: Die Eingabe wird mit jeder weiteren Nachricht länger. Dadurch werden bei jeder neuen Anfrage mehr Eingabe-Tokens benötigt, was die Kosten erhöht.

Anwendungen übermitteln deshalb häufig nur die letzten Nachrichten eines Gesprächs oder ersetzen ältere Nachrichten durch eine kurze Zusammenfassung.

Dadurch wird die Anzahl der Eingabe-Tokens begrenzt. Allerdings können dabei Informationen aus früheren Nachrichten verloren gehen.

## Schritt 3: Gleicher Prompt, unterschiedliche Antworten

Das Modell wählt das nächste Token anhand von Wahrscheinlichkeiten aus, ähnlich wie bei einem gewichteten Würfel.

Deshalb kann derselbe Prompt zu unterschiedlichen Antworten führen.

Wir senden dieselbe Anfrage fünfmal.

In [4]:
# The same prompt, five times
for i in [1, 2, 3, 4, 5]:
    r = client.responses.create(
        model=MODEL, input="Describe IKEA in exactly five words."
    )
    print("Run", i, ":", r.output_text)

Run 1 : Affordable flat-pack furniture from Sweden.
Run 2 : Affordable flat-pack furniture for everyone.
Run 3 : Affordable flat-pack furniture for everyone.
Run 4 : Affordable, flat-pack furniture for everyone.
Run 5 : Affordable, functional Scandinavian home furnishings.


## Schritt 4: Was kostet es, alle IKEA- und McDonald’s-Rezensionen zu analysieren?

Wir möchten jede Rezension mit folgendem Prompt an das Modell senden: «Positiv, negativ oder gemischt? Antworte mit einem Wort.»

Schätze die Kosten, **bevor** du etwas berechnest.

In [ ]:
# Your guess in USD (before you compute anything!)
my_guess_usd = 0.50  # TODO: e.g. 0.50


### Was ist ein Token?

Ein Modell verarbeitet Texte nicht direkt als einzelne Buchstaben oder Wörter, sondern als **Tokens.**

Tokens können aus vollständigen Wörtern, Wortbestandteilen oder Satzzeichen bestehen.

Die Kosten einer API-Anfrage hängen von der Anzahl der verwendeten Tokens ab.

`tiktoken`  ist ein Tokenizer für OpenAI-Modelle.

In [6]:
# Load the tokenizer
enc = tiktoken.get_encoding("o200k_base")

# A review from IKEA Philadelphia (1 of 5 stars, original spelling)
review = "Swedish meatballs good. Attitude at.Return/Exchange counter bad."

# Text -> token IDs (numbers)
token_ids = enc.encode(review)
print("Words: ", len(review.split()))
print("Tokens:", len(token_ids))
print("IDs:   ", token_ids)

# Token IDs -> pieces of text, one token at a time
pieces = []
for token_id in token_ids:
    pieces.append(enc.decode([token_id]))
print("Pieces:", pieces)

Words:  7
Tokens: 15
IDs:    [15254, 45708, 20872, 81079, 1899, 13, 11278, 5962, 540, 90846, 14, 35771, 9446, 4790, 13]
Pieces: ['Sw', 'edish', ' meat', 'balls', ' good', '.', ' Att', 'itude', ' at', '.Return', '/', 'Exchange', ' counter', ' bad', '.']


**Jetzt bist du dran:**

Probiere einen deutschen Satz, ein sehr langes Wort und einen Text mit Rechtschreibfehlern aus.

Was passiert mit der Anzahl der Tokens?

In [7]:
# TODO: replace the text and run the cell again
my_text = """Bei IKEA haben wir gefunden, was wir gesucht haben."""

# Text -> token IDs
token_ids = enc.encode(my_text)
print("Words: ", len(my_text.split()))
print("Tokens:", len(token_ids))

# Token IDs -> pieces of text
pieces = []
for token_id in token_ids:
    pieces.append(enc.decode([token_id]))
print("Pieces:", pieces)

Words:  9
Tokens: 11
Pieces: ['Bei', ' IKEA', ' haben', ' wir', ' gefunden', ',', ' was', ' wir', ' gesucht', ' haben', '.']


### Tokens aller IKEA- und McDonald’s-Rezensionen zählen

Die Zählung erfolgt lokal auf diesem Computer.

In diesem Schritt wird keine Rezension an die API gesendet. Deshalb können wir die Originaldatei verwenden.

In [8]:
# Read both complete datasets; counting is local, nothing is sent to the API
ikea = pd.read_csv(DATA / "ikea_reviews.csv")
mcdonalds = pd.read_csv(DATA / "mcdonalds_reviews.csv")
ikea["chain"] = "IKEA"
mcdonalds["chain"] = "McDonald's"
df = pd.concat([ikea, mcdonalds], ignore_index=True)

# Count the tokens of every review
tokens = []
for text in df["text"]:
    token_ids = enc.encode(text)
    tokens.append(len(token_ids))
df["tokens"] = tokens

# Reviews and text tokens per chain
budget = df.groupby("chain").agg(
    reviews=("text", "size"), text_tokens=("tokens", "sum")
)
budget

,reviews,text_tokens
chain,,
IKEA,1743,291397
McDonald's,18280,1846324


### Kosten berechnen und mit deiner Schätzung vergleichen

In [9]:
# Every review gets its own prompt
PROMPT = "Positive, negative or mixed? Answer with one word.\n"
prompt_tokens = len(enc.encode(PROMPT))

# Input tokens: review tokens plus one prompt per review
budget["input_tokens"] = (
    budget["text_tokens"] + budget["reviews"] * prompt_tokens
)

# Cost: tokens / 1 million x price per 1 million tokens
budget["input_cost_usd"] = (
    budget["input_tokens"] / 1_000_000 * PRICE_INPUT_PER_M
)
cost = budget["input_cost_usd"].sum()

print(budget.to_string())
print(f"\nBoth chains (input): {cost:.4f} USD")
print(f"Your guess:          {my_guess_usd} USD")

            reviews  text_tokens  input_tokens  input_cost_usd
chain                                                         
IKEA           1743       291397        310570        0.031057
McDonald's    18280      1846324       2047404        0.204740

Both chains (input): 0.2358 USD
Your guess:          None USD


Die grössere Unternehmenskette verfügt über mehr Rezensionen. Allerdings beeinflusst auch die Länge der einzelnen Rezensionen das benötigte Token-Budget.

Diese Berechnungen berücksichtigen sowohl die Rezensionstexte als auch den Prompt, der bei jeder Anfrage erneut mitgesendet wird.

Durch die Formatierung der API-Nachrichten entstehen zusätzliche Tokens.

Die Tokens der Antworten (Output-Tokens) kommen noch hinzu. Sie sind teurer als die Eingabe-Tokens (Input-Tokens).


### Kontextfenster (Context Window)

Das Kontextfenster bezeichnet die maximale Anzahl an Tokens, die innerhalb einer einzelnen Anfrage verarbeitet werden können. Dazu gehören sowohl die Eingabe als auch die Ausgabe.

Für diese Übung verwenden wir ein Kontext-Budget von 1'000'000 Tokens.

Reserviere dabei genügend Platz für den Prompt, die Nachrichtenformatierung und die Ausgabe sowie für die Rezensionstexte.

## Schritt 5: Token-Verbrauch und Reasoning-Aufwand
Sende dieselbe kurze Rezension einmal mit niedrigem und einmal mit hohem Reasoning-Aufwand (Denkaufwand) an das Modell.

Vergleiche dabei:
- Die Klassifizierung der Rezension (Label)
- Die Anzahl der Eingabe-Tokens
- Die Anzahl der Ausgabe-Tokens
- Die Unterschiede zur lokalen Token-Zählung

Die API-Nutzungsstatistik (Usage) zeigt ausserdem, wie viele Ausgabe-Tokens für den internen, nicht sichtbaren Denkprozess (Hidden Reasoning) verwendet wurden.

In [10]:
# Local token count of the same input, for comparison with the API usage
print("Local input-token count:", len(enc.encode(PROMPT + review)))

# Send the same input with low and high reasoning effort.
# store=False: the provider does not keep the request.
for effort in ["low", "high"]:
    r = client.responses.create(
        model=MODEL,
        input=PROMPT + review,
        reasoning={"effort": effort},
        store=False,
    )
    usage = r.usage
    print("Effort:", effort, "| answer:", r.output_text)
    print(
        "Input:",
        usage.input_tokens,
        "| output:",
        usage.output_tokens,
        "| of these, reasoning:",
        usage.output_tokens_details.reasoning_tokens,
    )

Local input-token count: 26
Effort: low | answer: Mixed
Input: 32 | output: 20 | of these, reasoning: 13
Effort: high | answer: Mixed
Input: 32 | output: 28 | of these, reasoning: 21


Die Token-Zählung der API berücksichtigt auch die Formatierung der Nachrichten.

Die Anzahl der Ausgabe-Tokens kann zudem interne Reasoning-Tokens enthalten, selbst wenn die sichtbare Antwort nur aus einem einzigen Wort besteht.

Ein höherer Reasoning-Aufwand kann die Anzahl der Ausgabe-Tokens und damit die Kosten erhöhen.

Ob dadurch tatsächlich eine nützliche Verbesserung der Antwortqualität entsteht, muss gemessen und überprüft werden.

## Überprüfe dein Verständnis

Beantworte die folgenden Fragen kurz anhand der oben erzeugten Ergebnisse.

Versuche zunächst, die Fragen selbst zu beantworten, und öffne anschliessend die Hinweise.

1. Warum kann sich das Modell in Schritt 2 auf eine frühere Frage beziehen? Bedeutet das, dass sich die Modellgewichte während des Gesprächs verändert haben?

2. Verwende ein Ergebnis der Tokenisierung, um zu erklären, warum das blosse Zählen von Wörtern nicht ausreicht, um die Eingabekosten abzuschätzen.

3. Warum muss bei der Kostenberechnung der Prompt für jede Rezension erneut berücksichtigt werden? Welche Kosten fehlen noch bei einer Schätzung, die ausschliesslich die Eingabe-Tokens berücksichtigt?

4. Warum können wiederholte Anfragen unterschiedliche Antworten liefern? Weshalb ist ein einzelner Durchlauf keine ausreichende Grundlage, um verschiedene Prompts miteinander zu vergleichen?

5. Wie kann eine Antwort, die nur aus einem Wort besteht, mehr als ein Ausgabe-Token verbrauchen? Wie könntest du feststellen, ob sich der höhere Reasoning-Aufwand im Verhältnis zu den zusätzlichen Kosten lohnt?

**Deine Antworten:**

1. Die Anwendung sendet den bisherigen Nachrichtenverlauf bei jeder neuen Anfrage erneut mit. Das Modell kann diese Informationen deshalb in der aktuellen Eingabe berücksichtigen. Die Modellgewichte verändern sich dabei nicht.

2. Tokens können vollständige Wörter, Wortbestandteile oder Satzzeichen sein. Ihre Anzahl hängt vom jeweiligen Text und vom verwendeten Tokenizer ab. Beziehe dich bei deiner Antwort auf die tatsächlich gemessenen Tokens.

3. Jede separate API-Anfrage enthält ihren eigenen Prompt. Deshalb muss dieser bei jeder Rezension erneut mitgezählt werden. Zusätzlich entstehen Kosten für Ausgabe-Tokens, einschliesslich interner Reasoning-Tokens. Auch die Nachrichtenformatierung benötigt zusätzliche Eingabe-Tokens.

4. Durch das probabilistische Sampling kann das Modell bei wiederholten Anfragen unterschiedliche nächste Tokens auswählen. Um Prompts sinnvoll zu vergleichen, sollten mehrere Durchläufe mit denselben Eingaben, demselben Modell und denselben Einstellungen durchgeführt werden.

5. Die API-Nutzungsstatistik kann interne Reasoning-Tokens sowie Satzzeichen in der Antwort berücksichtigen. Vergleiche bei mehreren Beispielen die gemessene Korrektheit der Antworten, die Anzahl der Ausgabe-Tokens und die Antwortzeit, um zu beurteilen, ob sich ein höherer Reasoning-Aufwand lohnt.

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hinweise zur Selbstkontrolle</summary>

1. Die Anwendung sendet den bisherigen Nachrichtenverlauf bei jeder neuen Anfrage erneut mit. Das Modell kann diese Informationen deshalb in der aktuellen Eingabe berücksichtigen. Die Modellgewichte verändern sich dabei nicht.

2. Tokens können vollständige Wörter, Wortbestandteile oder Satzzeichen sein. Ihre Anzahl hängt vom jeweiligen Text und vom verwendeten Tokenizer ab. Beziehe dich bei deiner Antwort auf die tatsächlich gemessenen Tokens.

3. Jede separate API-Anfrage enthält ihren eigenen Prompt. Deshalb muss dieser bei jeder Rezension erneut mitgezählt werden. Zusätzlich entstehen Kosten für Ausgabe-Tokens, einschliesslich interner Reasoning-Tokens. Auch die Nachrichtenformatierung benötigt zusätzliche Eingabe-Tokens.

4. Durch das probabilistische Sampling kann das Modell bei wiederholten Anfragen unterschiedliche nächste Tokens auswählen. Um Prompts sinnvoll zu vergleichen, sollten mehrere Durchläufe mit denselben Eingaben, demselben Modell und denselben Einstellungen durchgeführt werden.

5. Die API-Nutzungsstatistik kann interne Reasoning-Tokens sowie Satzzeichen in der Antwort berücksichtigen. Vergleiche bei mehreren Beispielen die gemessene Korrektheit der Antworten, die Anzahl der Ausgabe-Tokens und die Antwortzeit, um zu beurteilen, ob sich ein höherer Reasoning-Aufwand lohnt.

</details>

### Jupyter-Notebook – Informationen in der Fusszeile (bitte immer am Ende jedes Notebooks angeben)

In [11]:
# Show operating system, date/time and Python version
print("-----------------------------------")
print(os.name.upper())
print(platform.system(), "|", platform.release())
print("Datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Python Version:", python_version())
print("-----------------------------------")


-----------------------------------
POSIX
Linux | 6.8.0-1070-azure
Datetime: 2026-11-16 09:17:10
Python Version: 3.11.18
-----------------------------------
